# AutoNavLog Colab 操作プレビュー

AutoNavLogのUIと操作フローをColab上で確認するための**地上準備・プレビュー専用**Notebookです。出力は非公式の転記補助であり、運航資料・完成帳票・航空大学校の承認済み様式ではありません。

- 検証済みruntime ZIPをCLI検証時の `/content`、通常利用時のGoogle Drive `MyDrive` 直下の順に探します。ZIPには固定版のAutoNavLog/MSM wheel、出典確認済みの参照・SR22 G6性能データ、検証済みPzs地形cacheが入ります。
- 上空風・気温はMSM予報値です。インターネット接続と取得可能なForecast Runが必要です。
- QNHはMSM海面更正気圧とPzs地形cacheから求める**MSM推定QNH**です。公式飛行場気象の観測QNHではないため、必ず原票と照合してください。
- MSM推定QNHを取得・算出できない場合は値を補完せず、利用者が適切なQNHを確認して手入力します。
- Projectの保存先は一時VMの `/content/AutoNavLog-preview-data` で、ランタイム終了時に消えます。
- KMLの境界を飛行経路として使う判断、性能・WX・警告・公式原票との照合は利用者が行ってください。

## 操作手順

1. Colabで「ランタイム」→「すべてのセルを実行」を選びます。
2. Google EarthのKML/XMLを貼り付け、Line/Polygon、進行順、各Pointの役割を確認します。
3. FROM/TOを有効な参照データから選び、VREP、Arrival、必要なCPと関連Legを確定します。
4. DATE、ETD、燃料、Phase、ALT、QNH、既定値を確認してNAV LOGを計算します。
5. MSM推定QNHを公式原票と照合し、取得不能または不採用の場合は手動QNHを入力して再計算します。
6. Blockerを解消し、Warningは内容を確認して表示された確認欄を選びます。
7. Route、Phase、ALT、WX、WCA、GS、TAS、ETE、燃料、Arrival、CP、出典を照合し、
   転記可になった非公式補助表をA4横で印刷します。Pzs地形cacheはQNH推定だけに使い、SEA・障害物評価は本版の対象外です。
8. 「保存」「Snapshot」は一時VMだけへ保存され、ランタイム終了時に消えます。

## UIに表示される作業区分

- 開始・環境確認
- Projectの作成・読込
- コース取込・編集
- 飛行計画入力
- Forecast Run選択
- 計算実行
- 警告・未確定項目の解消
- 清書ビュー（A4横の非公式転記補助表）
- 保存・Snapshot作成


In [ ]:
#@title AutoNavLog Preview配布版を検証して準備
from __future__ import annotations

import ctypes.util
import hashlib
import importlib.metadata
import json
import os
import shutil
import stat
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path, PurePosixPath

BUNDLE_VERSION = "0.2.1"
BUNDLE_NAME = f"autonavlog-colab-preview-{BUNDLE_VERSION}.zip"
EXPECTED_BUNDLE_SHA256 = (
    "53d66666d8e88e08d3996f9c41d1725470a8c2639c3d9d13d37820201d326596"
)
LOCAL_BUNDLE = Path("/content") / BUNDLE_NAME
DRIVE_BUNDLE = Path("/content/drive/MyDrive") / BUNDLE_NAME
RUNTIME_ROOT = Path("/content/AutoNavLog-preview-runtime")
MAX_MEMBER_COUNT = 64
MAX_MEMBER_SIZE = 128 * 1024 * 1024
MAX_TOTAL_SIZE = 256 * 1024 * 1024


def _require(condition: bool, message: str) -> None:
    if not condition:
        raise RuntimeError(message)


def _safe_member_path(value: str) -> PurePosixPath:
    _require(value != "", "runtime ZIPに空のmember名があります。")
    _require("\\" not in value, f"runtime ZIPの区切りが不正です: {value!r}")
    _require(
        not any(ord(character) < 32 for character in value),
        f"runtime ZIPのmember名に制御文字があります: {value!r}",
    )
    path = PurePosixPath(value)
    _require(not path.is_absolute(), f"runtime ZIPに絶対pathがあります: {value!r}")
    _require(
        all(part not in {"", ".", ".."} for part in path.parts),
        f"runtime ZIPにpath traversalがあります: {value!r}",
    )
    _require(
        not any(":" in part for part in path.parts),
        f"runtime ZIPにdrive/URI形式のpathがあります: {value!r}",
    )
    return path


if LOCAL_BUNDLE.is_file():
    BUNDLE = LOCAL_BUNDLE
else:
    from google.colab import drive

    drive.mount("/content/drive")
    BUNDLE = DRIVE_BUNDLE
if not BUNDLE.is_file():
    raise FileNotFoundError(
        "AutoNavLog runtime ZIPが見つかりません。"
        f"確認先: {LOCAL_BUNDLE}, {DRIVE_BUNDLE}"
    )
BUNDLE_SHA256 = hashlib.sha256(BUNDLE.read_bytes()).hexdigest()
_require(
    BUNDLE_SHA256 == EXPECTED_BUNDLE_SHA256,
    f"runtime ZIP本体のSHA-256が一致しません: {BUNDLE_SHA256}",
)

staging = Path(tempfile.mkdtemp(prefix="AutoNavLog-preview-runtime-", dir="/content"))
try:
    with zipfile.ZipFile(BUNDLE) as archive:
        infos = archive.infolist()
        _require(0 < len(infos) <= MAX_MEMBER_COUNT, "runtime ZIPのmember数が不正です。")
        info_by_name = {}
        total_size = 0
        for info in infos:
            _require(
                not info.is_dir(),
                f"runtime ZIPに予期しないdirectoryがあります: {info.filename}",
            )
            _safe_member_path(info.filename)
            _require(
                info.filename not in info_by_name,
                f"runtime ZIPに重複memberがあります: {info.filename}",
            )
            _require(
                not (info.flag_bits & 0x1),
                f"runtime ZIPに暗号化memberがあります: {info.filename}",
            )
            unix_mode = (info.external_attr >> 16) & 0xFFFF
            _require(
                not stat.S_ISLNK(unix_mode),
                f"runtime ZIPにsymlinkがあります: {info.filename}",
            )
            _require(
                unix_mode == 0 or stat.S_ISREG(unix_mode),
                f"runtime ZIPに通常file以外があります: {info.filename}",
            )
            _require(
                0 <= info.file_size <= MAX_MEMBER_SIZE,
                f"runtime ZIPのmemberが大きすぎます: {info.filename}",
            )
            total_size += info.file_size
            info_by_name[info.filename] = info
        _require(total_size <= MAX_TOTAL_SIZE, "runtime ZIPの展開後sizeが上限を超えます。")
        _require(
            "bundle-manifest.json" in info_by_name,
            "runtime ZIPにbundle-manifest.jsonがありません。",
        )
        manifest_bytes = archive.read("bundle-manifest.json")
        try:
            BUNDLE_MANIFEST = json.loads(manifest_bytes.decode("utf-8"))
        except (UnicodeDecodeError, json.JSONDecodeError) as error:
            raise RuntimeError("bundle manifestが有効なUTF-8 JSONではありません。") from error
        _require(
            isinstance(BUNDLE_MANIFEST, dict),
            "bundle manifestはobjectである必要があります。",
        )
        _require(
            BUNDLE_MANIFEST.get("schema_version") == 1,
            "bundle manifest schemaが一致しません。",
        )
        _require(
            BUNDLE_MANIFEST.get("bundle_version") == BUNDLE_VERSION,
            "runtime ZIP versionが一致しません。",
        )
        file_entries = BUNDLE_MANIFEST.get("files")
        _require(
            isinstance(file_entries, list) and file_entries,
            "bundle manifestにfilesがありません。",
        )
        entries_by_name = {}
        verified_contents = {}
        for index, entry in enumerate(file_entries):
            _require(
                isinstance(entry, dict),
                f"bundle manifest files[{index}]がobjectではありません。",
            )
            raw_name = entry.get("path")
            _require(
                isinstance(raw_name, str),
                f"bundle manifest files[{index}]のpathが不正です。",
            )
            member = str(_safe_member_path(raw_name))
            _require(
                member != "bundle-manifest.json",
                "bundle manifestは自分自身をhash化できません。",
            )
            _require(
                member not in entries_by_name,
                f"bundle manifestに重複fileがあります: {member}",
            )
            _require(
                member in info_by_name,
                f"bundle manifestのfileがZIPにありません: {member}",
            )
            content = archive.read(member)
            size = entry.get("size")
            digest = entry.get("sha256")
            _require(
                isinstance(size, int) and not isinstance(size, bool),
                f"bundle file sizeが不正です: {member}",
            )
            _require(len(content) == size, f"bundle file sizeが一致しません: {member}")
            _require(
                isinstance(digest, str) and len(digest) == 64,
                f"bundle file SHA-256が不正です: {member}",
            )
            _require(
                hashlib.sha256(content).hexdigest() == digest.lower(),
                f"bundle file SHA-256が一致しません: {member}",
            )
            entries_by_name[member] = entry
            verified_contents[member] = content
        _require(
            set(info_by_name) == set(entries_by_name) | {"bundle-manifest.json"},
            "runtime ZIPのmember集合がbundle manifestと一致しません。",
        )
        distributions = BUNDLE_MANIFEST.get("distributions")
        _require(
            isinstance(distributions, dict),
            "bundle manifestにdistributionsがありません。",
        )
        expected_distributions = (
            ("autonavlog", "0.2.1"),
            ("jma-msm-wind", "0.2.1"),
        )
        for distribution, expected_version in expected_distributions:
            descriptor = distributions.get(distribution)
            _require(
                isinstance(descriptor, dict),
                f"bundle manifestに{distribution}がありません。",
            )
            wheel_name = descriptor.get("path")
            _require(
                isinstance(wheel_name, str) and wheel_name in entries_by_name,
                f"{distribution} wheel pathが不正です。",
            )
            _require(
                descriptor.get("version") == expected_version,
                f"{distribution} versionが一致しません。",
            )
            _require(
                descriptor.get("size") == entries_by_name[wheel_name]["size"],
                f"{distribution} size記録が一致しません。",
            )
            _require(
                descriptor.get("sha256")
                == entries_by_name[wheel_name]["sha256"],
                f"{distribution} hash記録が一致しません。",
            )
        runtime_data_root = BUNDLE_MANIFEST.get("runtime_data_root")
        _require(runtime_data_root == "data", "runtime data rootが一致しません。")
        weather_contract = BUNDLE_MANIFEST.get("weather")
        _require(isinstance(weather_contract, dict), "weather契約がありません。")
        terrain_member = "data/msm/terrain.npz"
        _require(
            weather_contract.get("qnh") == "MSM_ESTIMATED_QNH",
            "QNH契約がMSM推定QNHではありません。",
        )
        _require(
            weather_contract.get("pzs_terrain_included") is True,
            "Pzs地形cache同梱契約がありません。",
        )
        _require(
            weather_contract.get("terrain_path") == terrain_member,
            "Pzs地形cache pathが一致しません。",
        )
        _require(terrain_member in entries_by_name, "Pzs地形cacheがありません。")
        _require(
            weather_contract.get("terrain_sha256")
            == entries_by_name[terrain_member]["sha256"],
            "Pzs地形cache hash記録が一致しません。",
        )
        for member, content in verified_contents.items():
            destination = staging.joinpath(*PurePosixPath(member).parts)
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(content)
        (staging / "bundle-manifest.json").write_bytes(manifest_bytes)
    if RUNTIME_ROOT.is_symlink() or RUNTIME_ROOT.is_file():
        RUNTIME_ROOT.unlink()
    elif RUNTIME_ROOT.is_dir():
        shutil.rmtree(RUNTIME_ROOT)
    os.replace(staging, RUNTIME_ROOT)
except BaseException:
    shutil.rmtree(staging, ignore_errors=True)
    raise

if ctypes.util.find_library("eccodes") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "libeccodes0"], check=True)

distribution_paths = BUNDLE_MANIFEST["distributions"]
msm_wheel = RUNTIME_ROOT / distribution_paths["jma-msm-wind"]["path"]
autonavlog_wheel = RUNTIME_ROOT / distribution_paths["autonavlog"]["path"]
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "--no-input",
        "--force-reinstall",
        str(msm_wheel),
        str(autonavlog_wheel),
    ],
    check=True,
)
_require(
    importlib.metadata.version("autonavlog") == "0.2.1",
    "AutoNavLog distribution versionが一致しません。",
)
_require(
    importlib.metadata.version("jma-msm-wind") == "0.2.1",
    "jma-msm-wind distribution versionが一致しません。",
)
print(
    f"AutoNavLog 0.2.1 / jma-msm-wind 0.2.1 / "
    f"bundle: {BUNDLE.name} ({BUNDLE_SHA256})"
)


In [ ]:
#@title 参照・性能・MSM推定QNH providerを構築
from autonavlog.application import CalculationService, ProjectService
from autonavlog.performance import PerformanceRepository
from autonavlog.presentation import AutoNavLogApp
from autonavlog.storage import (
    AirportRepository,
    LocalProjectRepository,
    ReferenceDataCatalogRepository,
)
from autonavlog.weather.msm_adapter import MsmWeatherProvider

APP_DATA = RUNTIME_ROOT / BUNDLE_MANIFEST["runtime_data_root"]
reference_data = ReferenceDataCatalogRepository(
    "/content/AutoNavLog-preview-data/reference-data",
    bundled_default=APP_DATA / "reference" / "default",
)
airports = AirportRepository.from_reference_catalog(reference_data.open_active())
performance = PerformanceRepository.from_directory(APP_DATA / "performance")
performance.require_verified()
weather = MsmWeatherProvider(
    cache_dir="/content/AutoNavLog-preview-msm-cache",
    terrain_cache_path=APP_DATA / "msm" / "terrain.npz",
)
repository = LocalProjectRepository("/content/AutoNavLog-preview-data")
projects = ProjectService(repository)
calculation = CalculationService(airports, performance)
app = AutoNavLogApp(projects, calculation, weather, reference_data)

print("参照pack・性能・Pzs地形cacheを検証しました。MSM取得は計算ボタンを押すまで開始しません。")


In [ ]:
#@title AutoNavLog Preview UIを表示
# KML・PILOT・SHIP・FROM・TOは固定せず、利用者が確認して入力します。
print("KMLを貼り付け、ETD・高度・Polygonの境界/開始点/進行方向を確認してください。")
print(
    "風・気温=MSM予報、QNH=MSM推定QNH。"
    "取得不能または不採用の場合は、公式原票を確認して手動QNHを入力してください。"
)
print("Pzs地形cacheはMSM推定QNHだけに使用します。SEA・障害物評価は本版の対象外です。")
print("地上準備用の非公式転記補助です。運航資料・完成帳票として使用できません。")
app.render()
